# 02 — Completing the AlphaEarth representation at sample boundaries

The initial AlphaEarth table was missing values for 12 of the 26,597 modelling locations. This notebook determines whether those omissions reflect genuine gaps in AlphaEarth coverage or small geometric mismatches where a point lies on, or immediately beside, a polygon boundary.

Each unmatched point is compared with the nearest AlphaEarth source polygon in British National Grid coordinates. A nearest-polygon match is accepted only within 50 metres. This is a conservative distance ceiling for resolving boundary and sliver effects; it is not a general spatial interpolation rule. The original AlphaEarth file is left unchanged, and a separate complete table is produced for modelling.

## Summary of the output

All 12 unmatched locations are PTAL observations distributed across several boroughs. Their nearest-polygon distances average 5.18 metres, with a median of 4.22 metres and a maximum of 14.42 metres. These distances are well below the 50-metre ceiling and indicate local boundary mismatches rather than a systematic area of missing coverage.

After the nearest-polygon matches are added, AlphaEarth has complete coverage for all 26,597 samples. Each row retains a field showing whether the match was direct or used the boundary fallback, together with the fallback distance. These fields document data provenance and are not used as predictive features.

In [ ]:
# Connect Google Drive and load the spatial-analysis dependencies.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import re
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) not in sys.path:
    sys.path.append(str(FINAL_CODE_DIR))

from config import *

print("Sample:", DINO_SAMPLE_PATH, DINO_SAMPLE_PATH.exists())
print("Legacy AlphaEarth:", ALPHA_EMB_PATH, ALPHA_EMB_PATH.exists())
print("AlphaEarth polygons:", ALPHA_GPKG_PATH, ALPHA_GPKG_PATH.exists())
print("Final AlphaEarth:", ALPHA_FINAL_PATH)


## 1. Locate the unmatched samples

The sample table is joined to the original AlphaEarth embeddings by `sample_id`. Rows with any missing AlphaEarth dimension are isolated, and the expected 64-dimensional feature set is confirmed.

In [ ]:
# Join the sample to AlphaEarth and isolate rows without a complete 64-dimensional vector.

sample = pd.read_csv(DINO_SAMPLE_PATH)
alpha = pd.read_parquet(ALPHA_EMB_PATH)

sample["sample_id"] = sample["sample_id"].astype(str)
alpha["sample_id"] = alpha["sample_id"].astype(str)

alpha_cols = sorted(
    [c for c in alpha.columns if c.startswith("alphaearth_2024_")],
    key=lambda x: int(re.search(r"(\d+)$", x).group(1))
)
assert len(alpha_cols) == 64, f"Expected 64 AlphaEarth dimensions, got {len(alpha_cols)}"

check = sample.merge(
    alpha[["sample_id"] + alpha_cols],
    on="sample_id",
    how="left",
    validate="one_to_one"
)
check["alpha_missing"] = check[alpha_cols].isna().any(axis=1)
missing = check[check["alpha_missing"]].copy()

print("Total:", len(check))
print("Missing:", len(missing))
print("Missing %:", len(missing)/len(check)*100)
display(missing[["sample_id","task","x","y","borough"]])


## 2. Examine the geographical pattern of missingness

Counts by task and borough, together with a map of the unmatched points, show whether the omissions form a coherent coverage gap or are scattered boundary cases.

In [ ]:
# Summarise where the unmatched locations occur.

print("Missing by task")
display(missing["task"].value_counts(dropna=False).rename("n_missing").to_frame())

print("Missing by borough")
display(missing["borough"].value_counts(dropna=False).rename("n_missing").to_frame())

borough_audit = (
    check.groupby("borough", dropna=False)
    .agg(n_total=("sample_id","size"), n_missing=("alpha_missing","sum"))
    .reset_index()
)
borough_audit["missing_pct"] = borough_audit["n_missing"] / borough_audit["n_total"] * 100

city_missing = int(
    missing["borough"].astype(str).str.fullmatch("City of London", case=False, na=False).sum()
)
print("Missing in City of London:", city_missing)

display(borough_audit.sort_values(["n_missing","missing_pct"], ascending=False).head(20))


## 3. Read the source polygons and identify their feature columns

The AlphaEarth polygon layer is projected to British National Grid. Its 64 source dimensions are matched to the column names used in the sample-level embedding table.

In [ ]:
# Read and project the AlphaEarth source polygons and identify their feature columns.

alpha_poly = gpd.read_file(ALPHA_GPKG_PATH)
if alpha_poly.crs is None:
    raise ValueError("AlphaEarth polygon CRS is missing.")
alpha_poly = alpha_poly.to_crs("EPSG:27700")

# Prefer already-renamed columns if present; otherwise detect band1_mean ... band64_mean.
if all(col in alpha_poly.columns for col in alpha_cols):
    poly_feature_cols = alpha_cols
    rename_poly = {}
else:
    band_candidates = [
        col for col in alpha_poly.columns
        if re.fullmatch(r"band\d+_mean", str(col))
    ]
    band_candidates = sorted(
        band_candidates,
        key=lambda x: int(re.search(r"band(\d+)_mean", x).group(1))
    )
    if len(band_candidates) != 64:
        raise ValueError(
            f"Could not identify 64 AlphaEarth polygon feature columns. "
            f"Found {len(band_candidates)} band*_mean columns."
        )
    poly_feature_cols = band_candidates
    rename_poly = dict(zip(poly_feature_cols, alpha_cols))

print("Polygon rows:", len(alpha_poly))
print("Polygon feature columns:", len(poly_feature_cols))
print("CRS:", alpha_poly.crs)


## 4. Match each missing point to the nearest polygon

Only polygons near the 12 unmatched points are considered for efficiency. The nearest geometry and its distance are recorded, and every accepted match must lie within the 50-metre limit.

In [ ]:
# Find the nearest source polygon for each verified boundary case.

missing_gdf = gpd.GeoDataFrame(
    missing[["sample_id","task","x","y","borough"]].copy(),
    geometry=gpd.points_from_xy(missing["x"], missing["y"]),
    crs="EPSG:27700"
)

# Small bbox subset is only an efficiency optimisation.
minx, miny, maxx, maxy = missing_gdf.total_bounds
bbox_buffer_m = 5000
alpha_near = alpha_poly.cx[
    minx-bbox_buffer_m:maxx+bbox_buffer_m,
    miny-bbox_buffer_m:maxy+bbox_buffer_m
].copy()

fallback = gpd.sjoin_nearest(
    missing_gdf,
    alpha_near[poly_feature_cols + ["geometry"]],
    how="left",
    max_distance=ALPHA_NEAREST_MAX_DISTANCE_M,
    distance_col="alphaearth_nearest_dist_m"
)

# A geometric tie can produce more than one nearest polygon. Keep the smallest distance
# and then the first deterministic row for that sample.
fallback = (
    fallback
    .sort_values(["sample_id","alphaearth_nearest_dist_m","index_right"], kind="mergesort")
    .drop_duplicates("sample_id", keep="first")
    .copy()
)

if rename_poly:
    fallback = fallback.rename(columns=rename_poly)

fallback["alphaearth_match_method"] = "nearest_fallback"

display(
    fallback[[
        "sample_id","task","borough","alphaearth_nearest_dist_m"
    ]].sort_values("alphaearth_nearest_dist_m")
)

if fallback[alpha_cols].isna().any(axis=None):
    failed = fallback.loc[fallback[alpha_cols].isna().any(axis=1), "sample_id"].tolist()
    raise ValueError(f"Nearest fallback failed for samples: {failed}")

if len(fallback) != len(missing):
    raise ValueError(f"Expected {len(missing)} fallback rows, got {len(fallback)}")

print("Max fallback distance:",
      float(fallback["alphaearth_nearest_dist_m"].max()), "m")


## 5. Build the complete AlphaEarth table

Direct matches are retained as supplied. The 12 verified boundary cases receive the feature vector of their nearest polygon. Match method and distance are stored alongside the 64 representation dimensions.

In [ ]:
# Combine direct and nearest-polygon matches into a complete sample table.

alpha_final = (
    sample[["sample_id"]]
    .merge(alpha[["sample_id"] + alpha_cols], on="sample_id", how="left", validate="one_to_one")
)

alpha_final["alphaearth_match_method"] = np.where(
    alpha_final[alpha_cols].isna().any(axis=1),
    "nearest_fallback",
    "direct"
)
alpha_final["alphaearth_nearest_dist_m"] = np.where(
    alpha_final["alphaearth_match_method"].eq("direct"),
    0.0,
    np.nan
)

fallback_idx = fallback.set_index("sample_id")
final_idx = alpha_final.set_index("sample_id")

for col in alpha_cols:
    ids = fallback_idx.index
    final_idx.loc[ids, col] = fallback_idx[col]

final_idx.loc[fallback_idx.index, "alphaearth_nearest_dist_m"] = (
    fallback_idx["alphaearth_nearest_dist_m"]
)

alpha_final = final_idx.reset_index()

n_missing_final = int(alpha_final[alpha_cols].isna().any(axis=1).sum())
print("Final rows:", len(alpha_final))
print("Final missing rows:", n_missing_final)
print("Match method:")
display(alpha_final["alphaearth_match_method"].value_counts().rename("n").to_frame())

assert len(alpha_final) == len(sample)
assert n_missing_final == 0
assert alpha_final["sample_id"].is_unique

alpha_final.to_parquet(ALPHA_FINAL_PATH, index=False)
print("Saved:", ALPHA_FINAL_PATH)


## 6. Save supporting tables and maps

The completed embedding table, a row-level record of the 12 fallback matches, borough summaries and quality-assurance maps are saved for reproducibility.

In [ ]:
# Save row-level match information, summaries and maps.

audit_out = missing[[
    "sample_id","task","x","y","lon","lat","borough"
]].merge(
    fallback[["sample_id","alphaearth_nearest_dist_m"]],
    on="sample_id",
    how="left",
    validate="one_to_one"
)
audit_out["alphaearth_match_method"] = "nearest_fallback"

audit_out.to_csv(
    AUDIT_DIR / "alphaearth_nearest_fallback_audit.csv",
    index=False
)
borough_audit.to_csv(
    AUDIT_DIR / "alphaearth_missing_by_borough.csv",
    index=False
)

summary = {
    "n_total": int(len(sample)),
    "n_original_missing": int(len(missing)),
    "original_missing_pct": float(len(missing)/len(sample)*100),
    "n_city_of_london_missing": int(city_missing),
    "n_final_missing": int(n_missing_final),
    "fallback_max_distance_m": float(fallback["alphaearth_nearest_dist_m"].max()),
    "fallback_mean_distance_m": float(fallback["alphaearth_nearest_dist_m"].mean()),
    "fallback_median_distance_m": float(fallback["alphaearth_nearest_dist_m"].median()),
    "max_distance_allowed_m": float(ALPHA_NEAREST_MAX_DISTANCE_M),
}
with open(AUDIT_DIR / "alphaearth_final_audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# QA figure 1: all sample points + original unmatched points.
all_gdf = gpd.GeoDataFrame(
    sample.copy(),
    geometry=gpd.points_from_xy(sample["x"], sample["y"]),
    crs="EPSG:27700"
)
fig, ax = plt.subplots(figsize=(8,8))
all_gdf.plot(ax=ax, markersize=1, alpha=0.08)
missing_gdf.plot(ax=ax, markersize=40)
ax.set_title("Original AlphaEarth-unmatched samples (QA)")
ax.set_xlabel("Easting (EPSG:27700)")
ax.set_ylabel("Northing (EPSG:27700)")
plt.tight_layout()
qa_path = FINAL_FIGURE_DIR / "alphaearth_missing_samples_clean_qa.png"
plt.savefig(qa_path, dpi=220, bbox_inches="tight")
plt.show()

print("Saved:")
print(AUDIT_DIR / "alphaearth_nearest_fallback_audit.csv")
print(AUDIT_DIR / "alphaearth_final_audit_summary.json")
print(qa_path)


## Interpretation

Generic median imputation is unnecessary for AlphaEarth because every original omission can be tied to a real source polygon within 14.42 metres. The completed table therefore provides an observed polygon representation for every modelling location: a direct point-in-polygon match for ordinary cases and a documented nearest-polygon match for the 12 boundary cases.